# Hack-A-Roo 2026, AI for Quantum track
# Notebook 6: Barren plateau predictor

**Problem.** Variational quantum circuits are trained by gradient descent on their rotation angles. For many circuit families the variance of the gradient over random initializations shrinks exponentially with the qubit count, so a typical random starting point sits on a landscape that is flat to within the measurement precision and the optimizer goes nowhere. This is the barren plateau, and it is a main reason variational algorithms struggle at scale. Measuring the gradient variance directly costs a full simulation sweep. Can a model predict it from a description of the circuit?

**What this notebook already does.** Two experiments. The first is controlled: one entangling pattern, one depth, qubit counts from 2 to 8, local and global cost functions, with a bootstrap error bar on every variance estimate. That is the plateau, measured cleanly. The second is the predictor: it enumerates every combination of qubit count, depth, entangling pattern, and cost locality in a grid, samples a subset of distinct configurations, estimates the gradient variance of each by the exact parameter-shift rule, and trains a regressor from circuit descriptors to log variance, scored on configurations held out from the grid.

**Metric.** For the controlled sweep, the slope of log10 variance against qubit count, with error bars. For the predictor, mean absolute error in log10 variance on held-out configurations. One unit of error is a factor of ten in variance.

**Hypothesis to test.** In the controlled sweep, the global cost loses a constant factor per added qubit and the local cost loses less at this depth. The predictor reaches an error well below one order of magnitude from eight cheap descriptors. The notebook does not assume either; the sixty parameter samples per configuration give a noisy variance estimate, and the regressor cannot beat that noise floor.

**Deliverable for the poster.** The controlled sweep figure and the predicted-versus-measured figure, plus the slope and the held-out error. Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes one to two minutes. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# Circuit configuration grid. The predictor experiment samples DISTINCT configurations from the full grid.
QUBIT_COUNT_CHOICES = [2, 3, 4, 5, 6, 7, 8]
LAYER_COUNT_CHOICES = [1, 2, 3, 4, 6, 8, 10, 12]
ENTANGLING_PATTERN_CHOICES = ["none", "linear", "ring", "full"]
COST_LOCALITY_CHOICES = ["local", "global"]     # local: Z on qubit 0. global: Z on every qubit (the parity).
NUMBER_OF_CONFIGURATIONS = 240                   # distinct configurations sampled from the grid; the grid has 7 * 8 * 4 * 2 = 448

# Gradient variance estimate. For each configuration, sample this many random parameter vectors and compute
# the partial derivative with respect to ONE parameter by the parameter-shift rule.
PARAMETER_SAMPLES_PER_CONFIGURATION = 60
GRADIENT_PARAMETER_LAYER = 0        # which layer the differentiated parameter lives in. Python indexing: 0 is the first layer, -1 is the last.
GRADIENT_PARAMETER_QUBIT = 0        # which qubit's ry rotation in that layer

# Controlled plateau sweep: one pattern, one depth, every qubit count, both costs, with bootstrap error bars.
PLATEAU_SWEEP_PATTERN = "linear"
PLATEAU_SWEEP_LAYERS = 8
PLATEAU_SWEEP_SAMPLES = 60
BOOTSTRAP_RESAMPLES = 200

# Learning
TEST_FRACTION = 0.25
GRADIENT_BOOSTING_TREES = 300
GRADIENT_BOOSTING_DEPTH = 3

# Output
OUTPUT_PATH = "barren_plateau_dataset.pkl"
FIGURE_PATH = "barren_plateau_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, scikit-learn 1.8 and 1.9.
!pip install -q "qiskit[visualization]" "scikit-learn>=1.5" matplotlib

In [ ]:
import time
import pickle
import itertools
import numpy as np
import sklearn
import qiskit
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

random_generator = np.random.default_rng(RANDOM_SEED)
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "scikit-learn": sklearn.__version__}
print("imports done", package_versions)

In [ ]:
# The ansatz, built gate by gate from a flat angle array.
#
# One layer on n qubits: ry(angle) on every qubit, then rz(angle) on every qubit, then the entangling pattern in CZ.
# Parameters per layer = 2n. The angle array has shape (layers, 2n): row k holds [ry angles..., rz angles...].
#
# Entangling patterns:
#   none    no two-qubit gates. The state is a product state at every depth. This is an analytic control, not a
#           plateau-free case: for one layer and the global cost, C = product over qubits of cos(ry angle), and the
#           variance of its derivative over uniform angles is exactly 2^(-n). Global-cost suppression does not need
#           entanglement; it needs a cost that is a product of many terms.
#   linear  CZ on (0,1), (1,2), ..., (n-2, n-1)
#   ring    linear plus CZ on (n-1, 0)
#   full    CZ on every pair
#
# The cost function is the expectation of a Pauli observable in the final state:
#   local   Z on qubit 0 only, label 'I...IZ'
#   global  Z on every qubit, label 'Z...Z'
# (qiskit labels put qubit 0 at the RIGHT end)

def entangling_pairs(qubit_count, pattern):
    """Lists the qubit pairs the pattern entangles"""
    if pattern == "none":
        return []
    if pattern == "linear":
        return [(qubit_index, qubit_index + 1) for qubit_index in range(qubit_count - 1)]
    if pattern == "ring":
        pairs = [(qubit_index, qubit_index + 1) for qubit_index in range(qubit_count - 1)]
        if qubit_count > 2:
            pairs.append((qubit_count - 1, 0))
        return pairs
    if pattern == "full":
        return [(qubit_i, qubit_j) for qubit_i in range(qubit_count) for qubit_j in range(qubit_i + 1, qubit_count)]
    raise ValueError(f"unknown entangling pattern {pattern}")


def build_ansatz_circuit(qubit_count, layer_count, pattern, angles):
    """Builds the layered ry-rz-CZ ansatz with the given numeric angles"""
    assert angles.shape == (layer_count, 2 * qubit_count)
    circuit = QuantumCircuit(qubit_count)
    for layer_index in range(layer_count):
        for qubit_index in range(qubit_count):
            circuit.ry(angles[layer_index, qubit_index], qubit_index)
        for qubit_index in range(qubit_count):
            circuit.rz(angles[layer_index, qubit_count + qubit_index], qubit_index)
        for qubit_i, qubit_j in entangling_pairs(qubit_count, pattern):
            circuit.cz(qubit_i, qubit_j)
    return circuit


def cost_observable(qubit_count, locality):
    """The Pauli observable whose expectation is the cost"""
    if locality == "local":
        return SparsePauliOp("I" * (qubit_count - 1) + "Z")
    if locality == "global":
        return SparsePauliOp("Z" * qubit_count)
    raise ValueError(f"unknown cost locality {locality}")


def cost_value(qubit_count, layer_count, pattern, locality, angles):
    """Exact expectation of the cost observable in the ansatz state"""
    statevector = Statevector(build_ansatz_circuit(qubit_count, layer_count, pattern, angles))
    return statevector.expectation_value(cost_observable(qubit_count, locality)).real


def partial_derivative(qubit_count, layer_count, pattern, locality, angles, parameter_layer=GRADIENT_PARAMETER_LAYER):
    """Parameter-shift derivative of the cost with respect to the ry angle on GRADIENT_PARAMETER_QUBIT in parameter_layer (negative counts from the last layer)"""
    # For a gate exp(-i theta P / 2) with P a Pauli, d<C>/d theta = [<C>(theta + pi/2) - <C>(theta - pi/2)] / 2. Exact, not a finite difference.
    assert -layer_count <= parameter_layer < layer_count, f"parameter layer {parameter_layer} does not exist in a {layer_count}-layer circuit"
    shifted_up = angles.copy()
    shifted_down = angles.copy()
    shifted_up[parameter_layer, GRADIENT_PARAMETER_QUBIT] += np.pi / 2
    shifted_down[parameter_layer, GRADIENT_PARAMETER_QUBIT] -= np.pi / 2
    return 0.5 * (cost_value(qubit_count, layer_count, pattern, locality, shifted_up)
                  - cost_value(qubit_count, layer_count, pattern, locality, shifted_down))


def gradient_variance(qubit_count, layer_count, pattern, locality, samples, parameter_layer=GRADIENT_PARAMETER_LAYER):
    """Returns the array of `samples` partial derivatives at uniformly random angles; its variance is the plateau statistic"""
    derivatives = np.zeros(samples)
    for sample_index in range(samples):
        angles = random_generator.uniform(0, 2 * np.pi, size=(layer_count, 2 * qubit_count))
        derivatives[sample_index] = partial_derivative(qubit_count, layer_count, pattern, locality, angles, parameter_layer)
    return derivatives


example_angles = random_generator.uniform(0, 2 * np.pi, size=(2, 6))
print(build_ansatz_circuit(3, 2, "ring", example_angles).draw(fold=120))
print("cost (local):", round(cost_value(3, 2, "ring", "local", example_angles), 4))
print("derivative (local):", round(partial_derivative(3, 2, "ring", "local", example_angles), 4))

# Sanity check the parameter-shift rule against a finite difference on this example, same parameter.
finite_difference_step = 1e-5
nudged_up = example_angles.copy()
nudged_up[GRADIENT_PARAMETER_LAYER, GRADIENT_PARAMETER_QUBIT] += finite_difference_step
nudged_down = example_angles.copy()
nudged_down[GRADIENT_PARAMETER_LAYER, GRADIENT_PARAMETER_QUBIT] -= finite_difference_step
finite_difference = (cost_value(3, 2, "ring", "local", nudged_up) - cost_value(3, 2, "ring", "local", nudged_down)) / (2 * finite_difference_step)
assert abs(finite_difference - partial_derivative(3, 2, "ring", "local", example_angles)) < 1e-6
print("parameter-shift rule agrees with finite difference")

# Sanity check the analytic control: one layer, no entanglement, global cost, variance 2^(-n).
analytic_check_qubits = 4
analytic_check_samples = 2000
analytic_derivatives = gradient_variance(analytic_check_qubits, 1, "none", "global", analytic_check_samples)
# standard error of a sample variance: sqrt((fourth moment - variance^2) / samples)
analytic_standard_error = np.sqrt((np.mean(analytic_derivatives ** 4) - analytic_derivatives.var() ** 2) / analytic_check_samples)
print(f"product ansatz, 1 layer, global cost on {analytic_check_qubits} qubits: measured variance {analytic_derivatives.var():.4f} +- {analytic_standard_error:.4f}, analytic 2^-n = {2.0 ** -analytic_check_qubits:.4f}")

In [ ]:
# EXPERIMENT 1, the controlled plateau sweep. Fixed pattern and depth, every qubit count, both costs.
# Each variance estimate gets a bootstrap standard error so the plot shows how much of the scatter is sampling noise.

plateau_sweep = []     # one dict per (qubit count, locality)
start_time = time.time()
for qubit_count in QUBIT_COUNT_CHOICES:
    for locality in COST_LOCALITY_CHOICES:
        derivatives = gradient_variance(qubit_count, PLATEAU_SWEEP_LAYERS, PLATEAU_SWEEP_PATTERN, locality, PLATEAU_SWEEP_SAMPLES)
        bootstrap_log10_variances = np.zeros(BOOTSTRAP_RESAMPLES)
        for resample_index in range(BOOTSTRAP_RESAMPLES):
            resampled = random_generator.choice(derivatives, size=PLATEAU_SWEEP_SAMPLES, replace=True)
            bootstrap_log10_variances[resample_index] = np.log10(max(resampled.var(), 1e-16))
        plateau_sweep.append({
            "qubit_count": qubit_count, "locality": locality,
            "variance": derivatives.var(),
            "log10_variance": np.log10(max(derivatives.var(), 1e-16)),
            "log10_variance_standard_error": bootstrap_log10_variances.std(),
        })
    print(f"qubits {qubit_count} done, elapsed {time.time() - start_time:.0f}s")

print()
print(f"CONTROLLED SWEEP: {PLATEAU_SWEEP_PATTERN} pattern, {PLATEAU_SWEEP_LAYERS} layers, {PLATEAU_SWEEP_SAMPLES} parameter samples per point")
print(f"{'qubits':>8s}{'local log10 var':>18s}{'global log10 var':>19s}")
for qubit_count in QUBIT_COUNT_CHOICES:
    row = f"{qubit_count:8d}"
    for locality in COST_LOCALITY_CHOICES:
        entry = next(record for record in plateau_sweep if record["qubit_count"] == qubit_count and record["locality"] == locality)
        row += f"{entry['log10_variance']:12.2f} +- {entry['log10_variance_standard_error']:4.2f}"
    print(row)

# Slope of log10 variance against qubit count, by least squares, for each cost.
plateau_slopes = {}
for locality in COST_LOCALITY_CHOICES:
    qubit_values = np.array([record["qubit_count"] for record in plateau_sweep if record["locality"] == locality])
    log10_values = np.array([record["log10_variance"] for record in plateau_sweep if record["locality"] == locality])
    plateau_slopes[locality] = np.polyfit(qubit_values, log10_values, 1)[0]
    print(f"{locality} cost: slope {plateau_slopes[locality]:.3f} in log10 variance per qubit, a factor of {10 ** -plateau_slopes[locality]:.2f} lost per added qubit")
print("A slope of -0.30 is a factor of 2 per qubit. This is one pattern at one depth over 2 to 8 qubits, which is a trend in a small ensemble, not an asymptotic proof.")

In [ ]:
# EXPERIMENT 2, the predictor. Enumerate the grid, sample DISTINCT configurations, estimate each one's variance.
# This is the slow cell.

DESCRIPTOR_NAMES = ["qubit_count", "layer_count", "parameter_count", "two_qubit_gate_count",
                    "pattern_is_linear", "pattern_is_ring", "pattern_is_full", "cost_is_global"]

configuration_grid = list(itertools.product(QUBIT_COUNT_CHOICES, LAYER_COUNT_CHOICES, ENTANGLING_PATTERN_CHOICES, COST_LOCALITY_CHOICES))
assert NUMBER_OF_CONFIGURATIONS <= len(configuration_grid), f"the grid only has {len(configuration_grid)} configurations"
sampled_grid_positions = random_generator.choice(len(configuration_grid), size=NUMBER_OF_CONFIGURATIONS, replace=False)
print(f"grid has {len(configuration_grid)} configurations; sampling {NUMBER_OF_CONFIGURATIONS} distinct ones")

configuration_records = []     # one dict per configuration, for inspection
descriptor_rows = []
log10_variance = []

start_time = time.time()
for configuration_index, grid_position in enumerate(sampled_grid_positions):
    qubit_count, layer_count, pattern, locality = configuration_grid[grid_position]

    derivatives = gradient_variance(qubit_count, layer_count, pattern, locality, PARAMETER_SAMPLES_PER_CONFIGURATION)
    # deep global-cost circuits can push the variance below floating-point resolution; floor it so log10 stays finite
    variance = max(derivatives.var(), 1e-16)

    two_qubit_gate_count = layer_count * len(entangling_pairs(qubit_count, pattern))
    configuration_records.append({
        "qubit_count": qubit_count, "layer_count": layer_count, "pattern": pattern, "locality": locality,
        "variance": variance, "mean_derivative": derivatives.mean(),
    })
    descriptor_rows.append([
        qubit_count, layer_count, layer_count * 2 * qubit_count, two_qubit_gate_count,
        pattern == "linear", pattern == "ring", pattern == "full", locality == "global",
    ])
    log10_variance.append(np.log10(variance))

    if (configuration_index + 1) % 20 == 0:
        elapsed_seconds = time.time() - start_time
        remaining_seconds = elapsed_seconds / (configuration_index + 1) * (NUMBER_OF_CONFIGURATIONS - configuration_index - 1)
        print(f"{configuration_index + 1}/{NUMBER_OF_CONFIGURATIONS} configurations  elapsed {elapsed_seconds:.0f}s  remaining about {remaining_seconds:.0f}s")

descriptor_rows = np.array(descriptor_rows, dtype=float)
log10_variance = np.array(log10_variance)
assert descriptor_rows.shape == (NUMBER_OF_CONFIGURATIONS, len(DESCRIPTOR_NAMES))
assert len(np.unique(descriptor_rows, axis=0)) == NUMBER_OF_CONFIGURATIONS, "descriptor rows must be distinct"
print()
print("log10 variance ranges from", round(log10_variance.min(), 2), "to", round(log10_variance.max(), 2))
print("mean derivative across all configurations (should be near 0):", round(np.mean([record["mean_derivative"] for record in configuration_records]), 4))
print(f"with {PARAMETER_SAMPLES_PER_CONFIGURATION} samples the variance estimate itself carries about 0.08 of log10 noise, which is the floor no predictor can beat")

In [ ]:
# Regressor from descriptors to log10 variance, scored on configurations that are not in the training set.
# Every row is a distinct configuration, so a row split is a configuration split.

train_indices, test_indices = train_test_split(
    np.arange(NUMBER_OF_CONFIGURATIONS), test_size=TEST_FRACTION, random_state=RANDOM_SEED)
assert len(np.intersect1d(train_indices, test_indices)) == 0

variance_regressor = GradientBoostingRegressor(
    n_estimators=GRADIENT_BOOSTING_TREES, max_depth=GRADIENT_BOOSTING_DEPTH, random_state=RANDOM_SEED)
variance_regressor.fit(descriptor_rows[train_indices], log10_variance[train_indices])
predicted_log10_variance = variance_regressor.predict(descriptor_rows[test_indices])

absolute_error = np.abs(predicted_log10_variance - log10_variance[test_indices])
print(f"RESULTS ON {len(test_indices)} HELD-OUT CONFIGURATIONS (none of them appear in training)")
print(f"  mean absolute error in log10 variance: {absolute_error.mean():.3f}   (1.0 = off by a factor of ten)")
print(f"  median absolute error:                 {np.median(absolute_error):.3f}")
print(f"  R^2:                                   {variance_regressor.score(descriptor_rows[test_indices], log10_variance[test_indices]):.3f}")
print("  descriptor importances:")
for descriptor_name, importance in zip(DESCRIPTOR_NAMES, variance_regressor.feature_importances_):
    print(f"    {descriptor_name:22s} {importance:.3f}")

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "QUBIT_COUNT_CHOICES": QUBIT_COUNT_CHOICES,
    "LAYER_COUNT_CHOICES": LAYER_COUNT_CHOICES,
    "ENTANGLING_PATTERN_CHOICES": ENTANGLING_PATTERN_CHOICES,
    "COST_LOCALITY_CHOICES": COST_LOCALITY_CHOICES,
    "NUMBER_OF_CONFIGURATIONS": NUMBER_OF_CONFIGURATIONS,
    "PARAMETER_SAMPLES_PER_CONFIGURATION": PARAMETER_SAMPLES_PER_CONFIGURATION,
    "GRADIENT_PARAMETER_LAYER": GRADIENT_PARAMETER_LAYER,
    "GRADIENT_PARAMETER_QUBIT": GRADIENT_PARAMETER_QUBIT,
    "PLATEAU_SWEEP_PATTERN": PLATEAU_SWEEP_PATTERN,
    "PLATEAU_SWEEP_LAYERS": PLATEAU_SWEEP_LAYERS,
    "PLATEAU_SWEEP_SAMPLES": PLATEAU_SWEEP_SAMPLES,
    "BOOTSTRAP_RESAMPLES": BOOTSTRAP_RESAMPLES,
    "TEST_FRACTION": TEST_FRACTION,
    "GRADIENT_BOOSTING_TREES": GRADIENT_BOOSTING_TREES,
    "GRADIENT_BOOSTING_DEPTH": GRADIENT_BOOSTING_DEPTH,
    "DESCRIPTOR_NAMES": DESCRIPTOR_NAMES,
    "package_versions": package_versions,
}
dataset = {
    "config": config,
    "plateau_sweep": plateau_sweep,
    "plateau_slopes": plateau_slopes,
    "configuration_records": configuration_records,
    "descriptor_rows": descriptor_rows,
    "log10_variance": log10_variance,
    "train_indices": train_indices,
    "test_indices": test_indices,
    "predicted_log10_variance_on_test": predicted_log10_variance,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: left, the controlled sweep with error bars. Right, predicted versus measured log10 variance on held-out configurations.

figure, (left_axis, right_axis) = plt.subplots(1, 2, figsize=(12, 4.5))

for locality, marker in zip(COST_LOCALITY_CHOICES, ["o", "s"]):
    qubit_values = [record["qubit_count"] for record in plateau_sweep if record["locality"] == locality]
    log10_values = [record["log10_variance"] for record in plateau_sweep if record["locality"] == locality]
    error_values = [record["log10_variance_standard_error"] for record in plateau_sweep if record["locality"] == locality]
    left_axis.errorbar(qubit_values, log10_values, yerr=error_values, marker=marker, capsize=3,
                       label=f"{locality} cost, slope {plateau_slopes[locality]:.2f} per qubit")
left_axis.set_xlabel("qubit count")
left_axis.set_ylabel("log10 variance of one partial derivative")
left_axis.set_title(f"Controlled sweep: {PLATEAU_SWEEP_PATTERN} pattern, {PLATEAU_SWEEP_LAYERS} layers, bootstrap error bars")
left_axis.legend()

right_axis.scatter(log10_variance[test_indices], predicted_log10_variance, alpha=0.7)
diagonal = [log10_variance.min(), log10_variance.max()]
right_axis.plot(diagonal, diagonal, "k--", linewidth=0.8)
right_axis.set_xlabel("measured log10 variance")
right_axis.set_ylabel("predicted log10 variance")
right_axis.set_title(f"Predictor on {len(test_indices)} held-out configurations, MAE {absolute_error.mean():.2f}")

figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run measures one plateau cleanly and fits a predictor to a grid of small circuits. Neither is an asymptotic statement, and the predictor is only as good as the grid it saw. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. Report bootstrap error bars on every variance estimate, and if you sweep a setting for the predictor, choose it on a validation subset of configurations and report the test number once.

1. **Death depth.** For each qubit count and entangling pattern, sweep depth finely (1 to 30 layers) and find the depth at which the variance drops below a threshold such as 1e-4. Does the death depth shrink with qubit count, and does the full pattern die faster than linear? Metric: death depth versus qubit count, one line per pattern, with the threshold crossing bracketed by its error bars.

2. **Local versus global cost at shallow depth.** The theory for local costs gives guarantees at depth logarithmic in the qubit count for specific circuit families and random-initialization assumptions, and it does not cover every pattern here. Pin down empirically the shallow-depth regime where local survives and global dies for this ansatz. Metric: variance versus depth at 8 qubits, local and global on one plot, and the depth at which they separate.

3. **Which parameter you differentiate.** `partial_derivative` takes `parameter_layer`, and -1 means the last layer of whatever circuit it is given. Measure the gradient variance as a function of the parameter's layer position, holding qubit count, depth, and pattern fixed. A parameter next to the measurement has a small light cone to the measurement, but the state arriving at it has already been through every earlier layer, so it is an open question whether its gradient stays large; with the full pattern at 8 qubits a quick check gave about 0.5 at one layer and about 0.006 at twelve. Metric: variance versus parameter layer position at several depths, local and global cost.

4. **Extrapolate to sizes you cannot simulate.** Train the predictor on 2 to 6 qubits only and test it on 7 and 8. Does it extrapolate the exponential decay, or does it flatten out? Then try a model with the right inductive bias, such as linear regression on log variance with qubit count and depth as inputs. Metric: mean absolute error on the held-out sizes, gradient boosting versus linear.

5. **Initialization strategies.** The base run samples every angle uniformly in [0, 2pi]. Sample them instead from a narrow range around zero, or set each second layer to undo the first (identity-block initialization). Does the initial gradient variance recover, and for how many optimizer steps does that last? Metric: variance versus the initialization spread, and variance after 20 steps of gradient descent on a random target.

6. **Noise-induced plateau.** Add depolarizing noise to every gate (`qiskit_aer` density-matrix simulation, see notebook hackaroo26-4-learned-error-mitigation for the noise model). Noise flattens the landscape even for a local cost at shallow depth. Metric: variance versus depth at a fixed noise rate, noiseless and noisy, local cost.

7. **A graph model instead of descriptors.** Represent the circuit as a graph (qubits as nodes, CZ gates as edges, depth as a node feature) and train a small graph neural network to predict log variance. Does it beat the hand descriptors on held-out configurations, and does it generalize to a pattern it never saw, such as a star? Metric: mean absolute error on held-out configurations and on the unseen pattern.

Whatever you choose, load `barren_plateau_dataset.pkl` for the controlled sweep, the base-run configurations, descriptors, variances, and split. The `config` dict inside tells you exactly which run produced it.

In [ ]:
# Your extension starts here.
